# Chapter 5 – Worked examples

This notebook runs the computations of [Chapter 5](../index.md) live: the clipped Poisson count of mutated
substations, exactly and by simulation; the mean of the clipped variable against the upper clip; seeds and
independent streams in NumPy and JAX; the discretized Gaussian of PST tap steps; the maximum of uniforms; and
thirty runs of two optimizer variants. The code lives in the modules next to it (`clipped_poisson.py`,
`rng_streams.py`, `pst_steps.py`, `uniform_max.py`, `two_variants.py`); the cells only call them.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import clipped_poisson as cp
import pst_steps as ps
import rng_streams as rs
import two_variants as tv
import uniform_max as um

## The clipped Poisson count $K = \min(\max(N, 1), 3)$, $N \sim \text{Poisson}(2)$

The exact probabilities absorb $P(N = 0) = e^{-2}$ into $K = 1$ and $P(N \ge 3)$ into $K = 3$; the mean of $K$ is
below the unclipped mean 2.

In [ ]:
print(cp.clipped_pmf(2.0, 3).round(4).to_string())
print(f'E[K] = {cp.clipped_mean(2.0, 3):.4f}    E[max(N, 1)] = {cp.mean_of_max_with_one(2.0):.4f}')

Ten thousand draws reproduce the probabilities to within about one percentage point.

In [ ]:
n, k = cp.simulate(seed=0)
print(cp.frequencies(k).round(4).to_string())
fig = cp.plot_frequencies(n, k, 2.0, 3)

## The mean of $K$ against the upper clip $m$

For $m = 1$ the mean is 1; it rises with $m$ towards $\lambda + e^{-\lambda} = 2.135$, the mean with the lower clip
alone, crossing $\lambda = 2$ between $m = 3$ and $m = 4$.

In [ ]:
table = cp.sample_mean_vs_m()
print(table.round(4).to_string(index=False))
fig = cp.plot_mean_vs_m(table, 2.0)

## Seeds and independent streams

The same seed replays the same draws; `spawn` gives independent children; JAX splits a key into fresh keys before
drawing, as `mutate.py` does with `substation_key` and `disconnection_key`.

In [ ]:
rs.main()

## PST tap steps from a discretized Gaussian ($\sigma = 1$)

In [ ]:
print(ps.step_distribution().round(4).to_string())
print(f'P(no move) = {ps.probability_no_move():.4f}, fraction of PSTs changing tap = {ps.fraction_changing():.4f}')

## The maximum of $n$ uniforms: $E[\max] = n / (n + 1)$

In [ ]:
print(um.comparison().to_string(index=False, float_format=lambda x: f'{x:.5f}'))

## Two optimizer variants, thirty runs each

Variant B is worse by 3 on average, with a run-to-run standard deviation of 10. One run per variant orders them
correctly only 58 % of the time; thirty runs each may or may not reach significance, depending on the seed.

In [ ]:
runs = tv.simulate_runs(0)
print(tv.summary(runs).round(2).to_string())
print(f'P(one run of A beats one run of B) = {tv.probability_single_run_orders_correctly():.3f}')
t, p = tv.welch_test(runs)
print(f'Welch t = {t:.2f}, p = {p:.3f}')
fig = tv.plot_ecdfs(runs)